In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import json

# Load the data from JSON
with open('/content/drive/MyDrive/ASD Level/Metadata/data.json', 'r') as f:
    data_json = json.load(f)

# Convert the 'TCImages' part of the JSON to a DataFrame
df = pd.DataFrame(data_json['TSImages'])
df = df.sort_values('ParticipantID').reset_index(drop = True)
df

In [ ]:
# Add the 'Label' column based on the CARS value
df['Label'] = df['CARS'].apply(lambda x: 0 if x < 30 else 1)

df.head()

In [ ]:
# Perform train-test split (30% test data)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Label'])

print("Train set size:", len(train_df))
print("Test set size:", len(test_df))

In [ ]:
from sklearn.model_selection import train_test_split

# Separate the dataset into the two classes
df_class_0 = df[df['Label'] == 0]
df_class_1 = df[df['Label'] == 1]

# Allocate 10 samples from class 0 to the test set
test_class_0, train_class_0 = train_test_split(df_class_0, test_size=len(df_class_0) - 10, random_state=42, stratify=df_class_0['Label'])

# Allocate 33 samples from class 1 to the test set
test_class_1, train_class_1 = train_test_split(df_class_1, test_size=len(df_class_1) - 33, random_state=42, stratify=df_class_1['Label'])

# Combine the test sets
test_df = pd.concat([test_class_0, test_class_1])

# Combine the remaining samples to form the training set
train_df = pd.concat([train_class_0, train_class_1])

# Shuffle the train and test sets to ensure random distribution
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Train set size:", len(train_df))
print("Class distribution in train set:")
print(train_df['Label'].value_counts())

print("Test set size:", len(test_df))
print("Class distribution in test set:")
print(test_df['Label'].value_counts())


In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class TSImagesDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.dataframe.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = self.dataframe.iloc[idx, -1]  # Assuming 'Label' is the last column

        if self.transform:
            image = self.transform(image)

        return image, label

# Define the image directory and transformations
img_dir = '/content/drive/MyDrive/ASD Level/Images/TSImages'
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 100x100 pixels
    transforms.ToTensor()  # Convert images to PyTorch tensors
])

# Create the training and testing datasets
train_dataset = TSImagesDataset(dataframe=train_df, img_dir=img_dir, transform=transform)
test_dataset = TSImagesDataset(dataframe=test_df, img_dir=img_dir, transform=transform)

# Create DataLoaders for training and testing sets
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Example: Iterate over the training DataLoader
for images, labels in train_dataloader:
    print("Train batch:")
    print(images.shape)  # Should print torch.Size([32, 3, 100, 100]) for a batch of 32 images
    print(labels)        # Labels for the batch
    break  # Just to test a single batch

# Example: Iterate over the testing DataLoader
for images, labels in test_dataloader:
    print("Test batch:")
    print(images.shape)  # Should print torch.Size([32, 3, 100, 100]) for a batch of 32 images
    print(labels)        # Labels for the batch
    break  # Just to test a single batch


## Resnet 18

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Load the pretrained ResNet18 model
model = models.resnet18(pretrained=True)

# Freeze the early layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer to match the number of classes in your task
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Assuming 2 classes (label 1 and label 2)

# Move the model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training the model
num_epochs = 1
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Print statistics
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(dataloader):.4f}")

print("Finished Training")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on the test set
model.eval()  # Set model to evaluation mode
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)

        # Get predictions
        _, preds = torch.max(outputs, 1)

        # Store labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")



In [ ]:
pd.DataFrame([all_labels, all_preds])

## Resnet 50

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Load the pretrained ResNet50 model
model = models.resnet50(pretrained=True)

# Freeze the early layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer to match the number of classes in your task
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Assuming 2 classes (label 0 and label 1)

# Move the model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training the model
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

print("Finished Training")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on the test set
model.eval()  # Set model to evaluation mode
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)

        # Get predictions
        _, preds = torch.max(outputs, 1)

        # Store labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")

## DenseNet121

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Load the pretrained DenseNet121 model
model = models.densenet121(pretrained=True)

# Freeze the early layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer to match the number of classes in your task
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 2)  # Assuming 2 classes (label 0 and label 1)

# Move the model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# Training the model
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

print("Finished Training")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on the test set
model.eval()  # Set model to evaluation mode
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)

        # Get predictions
        _, preds = torch.max(outputs, 1)

        # Store labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")



In [ ]:
pip install -q torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Load the pretrained Vision Transformer (ViT-B/16) model
model = models.vit_b_16(pretrained=True)

# Freeze the early layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer to match the number of classes in your task
num_ftrs = model.heads.head.in_features
model.heads.head = nn.Linear(num_ftrs, 2)  # Assuming 2 classes (label 0 and label 1)

# Move the model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.heads.head.parameters(), lr=0.001)

# Training the model
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

print("Finished Training")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on the test set
model.eval()  # Set model to evaluation mode
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)

        # Get predictions
        _, preds = torch.max(outputs, 1)

        # Store labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class TSImagesDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.dataframe.iloc[idx, 0])
        image = Image.open(img_name).convert('L')  # Convert image to grayscale
        label = self.dataframe.iloc[idx, -1]  # Assuming 'Label' is the last column

        if self.transform:
            image = self.transform(image)

        return image, label

# Define the image directory and transformations
img_dir = '/content/drive/MyDrive/ASD Level/Images/TSImages'
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224 pixels (to match ViT input size)
    transforms.ToTensor()  # Convert images to PyTorch tensors
])

# Create the training and testing datasets
train_dataset = TSImagesDataset(dataframe=train_df, img_dir=img_dir, transform=transform)
test_dataset = TSImagesDataset(dataframe=test_df, img_dir=img_dir, transform=transform)

# Create DataLoaders for training and testing sets
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Example: Iterate over the training DataLoader
for images, labels in train_dataloader:
    print("Train batch:")
    print(images.shape)  # Should print torch.Size([32, 1, 224, 224]) for a batch of 32 grayscale images
    print(labels)        # Labels for the batch
    break  # Just to test a single batch

# Example: Iterate over the testing DataLoader
for images, labels in test_dataloader:
    print("Test batch:")
    print(images.shape)  # Should print torch.Size([32, 1, 224, 224]) for a batch of 32 grayscale images
    print(labels)        # Labels for the batch
    break  # Just to test a single batch


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Load the pretrained ResNet18 model
model = models.resnet18(pretrained=True)

# Modify the first convolutional layer to accept 1-channel (grayscale) input instead of 3-channel (RGB) input
model.conv1 = nn.Conv2d(1, model.conv1.out_channels, kernel_size=model.conv1.kernel_size, stride=model.conv1.stride, padding=model.conv1.padding, bias=False)

# Freeze the early layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer to match the number of classes in your task
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Assuming 2 classes (label 0 and label 1)

# Move the model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training the model
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

print("Finished Training")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on the test set
model.eval()  # Set model to evaluation mode
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)

        # Get predictions
        _, preds = torch.max(outputs, 1)

        # Store labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Load the pretrained Vision Transformer (ViT-B/16) model
model = models.vit_b_16(pretrained=True)

# Modify the first convolutional layer to accept 1-channel (grayscale) input instead of 3-channel (RGB) input
model.conv_proj = nn.Conv2d(1, model.conv_proj.out_channels, kernel_size=model.conv_proj.kernel_size, stride=model.conv_proj.stride, padding=model.conv_proj.padding, bias=False)

# Freeze the early layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer to match the number of classes in your task
num_ftrs = model.heads.head.in_features
model.heads.head = nn.Linear(num_ftrs, 2)  # Assuming 2 classes (label 0 and label 1)

# Move the model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.heads.head.parameters(), lr=0.001)

# Training the model
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

print("Finished Training")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on the test set
model.eval()  # Set model to evaluation mode
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)

        # Get predictions
        _, preds = torch.max(outputs, 1)

        # Store labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import ViTForImageClassification, ViTFeatureExtractor
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os

# Load the pretrained ViT model from Hugging Face
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k', num_labels=2)

# Load the feature extractor for ViT
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')

# Modify dataset class to handle grayscale images and use the feature extractor
class TSImagesDataset(Dataset):
    def __init__(self, dataframe, img_dir, feature_extractor):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.dataframe.iloc[idx, 0])
        image = Image.open(img_name).convert('L')  # Convert image to grayscale
        image = image.convert('RGB')  # Convert grayscale to RGB by stacking

        # Apply the feature extractor (which includes resizing and normalization)
        image = self.feature_extractor(images=image, return_tensors="pt")['pixel_values'][0]

        label = self.dataframe.iloc[idx, -1]  # Assuming 'Label' is the last column
        return image, label

# Define the image directory
img_dir = '/content/drive/MyDrive/ASD Level/Images/TSImages'

# Create the datasets and dataloaders
train_dataset = TSImagesDataset(dataframe=train_df, img_dir=img_dir, feature_extractor=feature_extractor)
test_dataset = TSImagesDataset(dataframe=test_df, img_dir=img_dir, feature_extractor=feature_extractor)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the model
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(pixel_values=inputs).logits
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

print("Finished Training")



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Set the model to evaluation mode
model.eval()

# Initialize lists to store labels and predictions
all_labels = []
all_preds = []

# Disable gradient calculation for evaluation
with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass to get outputs
        outputs = model(pixel_values=inputs).logits

        # Get the predicted class (0 or 1)
        _, preds = torch.max(outputs, 1)

        # Store the true labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate evaluation metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

# Print the results
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import numpy as np
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler

# Custom Dataset Class
class TSImagesDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.dataframe.iloc[idx, 0])
        image = Image.open(img_name).convert('L')  # Convert image to grayscale
        if self.transform:
            image = self.transform(image)
        label = self.dataframe.iloc[idx, -1]  # Assuming 'Label' is the last column
        return image, label

# Define the image directory and transformations
img_dir = '/content/drive/MyDrive/ASD Level/Images/TSImages'
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224 pixels
    transforms.ToTensor(),  # Convert images to PyTorch tensors
    transforms.Normalize((0.5,), (0.5,))  # Normalize grayscale images
])

# Create the training and testing datasets
train_dataset = TSImagesDataset(dataframe=train_df, img_dir=img_dir, transform=transform)
test_dataset = TSImagesDataset(dataframe=test_df, img_dir=img_dir, transform=transform)

# Compute class weights for balancing
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_df['Label']), y=train_df['Label'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to('cuda')

# Create DataLoaders for training and testing sets
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Assuming train_dataloader and test_dataloader are already defined

# 1. Autoencoder Definition (Encoder only)
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # (B, 16, 112, 112)
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),  # (B, 32, 56, 56)
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # (B, 64, 28, 28)
            nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # (B, 128, 14, 14)
            nn.ReLU(True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),  # (B, 256, 7, 7)
            nn.ReLU(True)
        )

    def forward(self, x):
        x = self.encoder(x)
        return x

# Instantiate the encoder
encoder = Encoder().to('cuda')

# 2. ResNet50 Definition (Modified to accept encoded input)
class ResNet50Modified(nn.Module):
    def __init__(self, num_classes=2):
        super(ResNet50Modified, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        # Modify the ResNet to accept the output from the encoder directly
        self.resnet.conv1 = nn.Conv2d(256, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        x = self.resnet(x)
        return x

# Instantiate the modified ResNet50
model = ResNet50Modified().to('cuda')

# 3. Define the loss function and optimizer
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_df['Label']), y=train_df['Label'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to('cuda')
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Training the model
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to('cuda'), labels.to('cuda')

        # Encode the inputs
        encoded_inputs = encoder(inputs)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(encoded_inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

print("Finished Training")

# 5. Evaluate on the test set
model.eval()
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to('cuda'), labels.to('cuda')

        # Encode the inputs
        encoded_inputs = encoder(inputs)

        # Forward pass
        outputs = model(encoded_inputs)

        # Get the predicted class (0 or 1)
        _, preds = torch.max(outputs, 1)

        # Store the true labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate evaluation metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

# Print the results
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")
